# Lab 8: Part-of-Speech (POS) Tagging

In [1]:
import sys
import subprocess

import pandas as pd
import spacy
from spacy import displacy

# download the small English model only if it is missing
if not spacy.util.is_package('en_core_web_sm'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

nlp = spacy.load('en_core_web_sm')

## Task 1

In [2]:
sen = nlp(u"I like watching football matches. I do not know how to play it though")

play = sen[12]    # the word "play"
print(f'Word: "{play.text}"')
print()
print('1. POS tag of "play"                :', play.pos_)
print('2. Fine-grained POS tag of "play"   :', play.tag_)
print('3. Meaning of the fine-grained tag  :', spacy.explain(play.tag_))

Word: "play"

1. POS tag of "play"                : VERB
2. Fine-grained POS tag of "play"   : VB
3. Meaning of the fine-grained tag  : verb, base form


In [3]:
# 4. text, coarse-grained POS, fine-grained POS and explanation for every word
print(f"{'TEXT':<12} {'POS':<10} {'TAG':<8} EXPLANATION")
for word in sen:
    print(f'{word.text:{12}} {word.pos_:{10}} {word.tag_:{8}} {spacy.explain(word.tag_)}')

TEXT         POS        TAG      EXPLANATION
I            PRON       PRP      pronoun, personal
like         VERB       VBP      verb, non-3rd person singular present
watching     VERB       VBG      verb, gerund or present participle
football     NOUN       NN       noun, singular or mass
matches      NOUN       NNS      noun, plural
.            PUNCT      .        punctuation mark, sentence closer
I            PRON       PRP      pronoun, personal
do           AUX        VBP      verb, non-3rd person singular present
not          PART       RB       adverb
know         VERB       VB       verb, base form
how          SCONJ      WRB      wh-adverb
to           PART       TO       infinitival "to"
play         VERB       VB       verb, base form
it           PRON       PRP      pronoun, personal
though       ADV        RB       adverb


## Task 2

In [4]:
sentences = [(u"Did you search on Google?", "google"),
             (u"Let us google it.", "google"),
             (u"The has placed the food on the table", "table"),
             (u"She plans to table a motion for debate in Parliament", "table")]

rows = []
for n, (text, target) in enumerate(sentences, start=1):
    sen = nlp(text)
    word = next(token for token in sen if token.text.lower() == target)
    rows.append({'sentence': n, 'text': text, 'word': word.text,
                 'POS': word.pos_, 'TAG': word.tag_, 'explanation': spacy.explain(word.tag_)})

pd.DataFrame(rows).set_index('sentence')

,text,word,POS,TAG,explanation
sentence,,,,,
1,Did you search on Google?,Google,PROPN,NNP,"noun, proper singular"
2,Let us google it.,google,VERB,VB,"verb, base form"
3,The has placed the food on the table,table,NOUN,NN,"noun, singular or mass"
4,She plans to table a motion for debate in Parl...,table,VERB,VB,"verb, base form"


**Comparison**

The same word receives a different tag depending on how it is used in the sentence:

- **google**: *Google* in "Did you search on Google?" is a **proper noun** (`NNP`: noun, proper singular), but *google* in "Let us google it." is a **verb** (`VB`: verb, base form).
- **table**: *table* in "...food on the table" is a **noun** (`NN`: noun, singular or mass), but *table* in "She plans to table a motion..." is a **verb** (`VB`: verb, base form).

So spaCy does not look the word up in a dictionary: it predicts the tag from the context (e.g. the words around it such as *us*, *to* and *the*).

## Task 3

In [5]:
doc = nlp(u"I like to play football. I hated it in my childhood though")

displacy.render(doc, style='dep', jupyter=True, options={'distance': 110})